## 1. Environment Setup & Imports

Initializing the research environment. We import standard data manipulation libraries (`pandas`, `numpy`), statistical tools (`scipy.stats` for Spearman rank correlation), and `plotly` for high-quality, interactive financial visualizations. We also configure the default Plotly template to fit a dark-themed quant research style.

In [1]:
# ==========================================
# CELL 1: SETUP & IMPORTS
# ==========================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import spearmanr
import warnings

# Ignore pandas warnings for a cleaner output
warnings.filterwarnings('ignore')

# Set default aesthetic for Plotly (Quant style)
import plotly.io as pio
pio.templates.default = "plotly_dark"

print("Environment ready. Libraries imported.")

Environment ready. Libraries imported.


## 2. Data Loading, Cleaning & Timezone Synchronization

This step is critical for cross-asset analysis to avoid look-ahead bias and misalignment:
* **BTCUSDT** trades 24/7, and its data exported from Binance is natively in `UTC`.
* **NQ Futures (MNQ)** trade on the CME. The IBKR data is in the `US/Central` timezone (Chicago) and includes weekend gaps and intraday maintenance breaks.

We localize the NQ data to Chicago time (handling Daylight Saving Time transitions automatically, like the one on March 10, 2024) and convert it to `UTC`. Finally, an `inner join` automatically removes crypto weekend noise, leaving us with a perfectly synchronized dataset containing only regular futures trading hours.

In [2]:
# ==========================================
# CELL 2: DATA LOADING, CLEANING & SYNC
# ==========================================

# 1. Define local file paths
BTC_PATH = r"C:\Python\studia\quant_connect\Correlation_Regime_Strategy\new\BTCUSDT_1m_2024-03-07_to_2026-03-07.csv"
NQ_PATH = r"C:\Python\algo\data\NQ_stitched_1min_2024-03-07_to_2026-03-07.csv"

# 2. Load and process BTC data (Binance)
# Binance exports dates in UTC natively
df_btc = pd.read_csv(BTC_PATH)
df_btc['Open Time'] = pd.to_datetime(df_btc['Open Time'])
df_btc.set_index('Open Time', inplace=True)
df_btc.index = df_btc.index.tz_localize('UTC')

# Keep only essential columns and standardize naming
df_btc = df_btc[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df_btc.columns = ['btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume']

# 3. Load and process NQ data (IBKR)
# Recognized timezone is US/Central (Chicago) Exchange Time
df_nq = pd.read_csv(NQ_PATH)
df_nq['date'] = pd.to_datetime(df_nq['date'], format='%Y%m%d %H:%M:%S')
df_nq.set_index('date', inplace=True)

# CRITICAL: Timezone localization and conversion to UTC
# We use ambiguous='NaT' and nonexistent='shift_forward' to handle the 
# DST spring forward transition smoothly without throwing errors.
df_nq.index = df_nq.index.tz_localize(
    'America/Chicago', 
    ambiguous='NaT', 
    nonexistent='shift_forward'
).tz_convert('UTC')

df_nq = df_nq[['open', 'high', 'low', 'close', 'volume']].copy()
df_nq.columns = ['nq_open', 'nq_high', 'nq_low', 'nq_close', 'nq_volume']

# 4. Synchronization (Inner Join)
# This aligns the timestamps and drops BTC data during NQ weekend/maintenance gaps
df = df_nq.join(df_btc, how='inner')

# 5. Calculate Returns
# Simple percentage returns are sufficient for our directional correlation metrics
df['btc_ret'] = df['btc_close'].pct_change()
df['nq_ret'] = df['nq_close'].pct_change()

# Drop the first row (NaN from pct_change)
df.dropna(inplace=True)

print(f"Data synchronized successfully. Total common observations (1m candles): {len(df)}")
display(df.head())

Data synchronized successfully. Total common observations (1m candles): 682062


,nq_open,nq_high,nq_low,nq_close,nq_volume,btc_open,btc_high,btc_low,btc_close,btc_volume,btc_ret,nq_ret
2024-03-07 23:01:00+00:00,18526.50,18527.00,18516.25,18517.75,41,67128.39,67139.99,67102.00,67102.00,19.44230,-0.000393,-0.000540
2024-03-07 23:02:00+00:00,18515.75,18515.75,18510.75,18510.75,21,67102.00,67102.01,67061.20,67065.39,24.32207,-0.000546,-0.000378
2024-03-07 23:03:00+00:00,18511.25,18518.25,18506.25,18508.75,47,67065.39,67078.00,67060.18,67060.95,20.91589,-0.000066,-0.000108
2024-03-07 23:04:00+00:00,18511.75,18512.00,18510.00,18510.75,10,67060.94,67078.06,67023.80,67053.74,22.90025,-0.000108,0.000108
2024-03-07 23:05:00+00:00,18511.00,18512.50,18509.25,18511.50,17,67053.74,67100.96,67053.74,67100.96,21.01717,0.000704,0.000041


## 3. Correlation Metrics Engine (The Math)

In this section, we calculate three distinct metrics over a rolling window to identify true structural alignment between BTC and NQ. 

1. **Rolling Pearson Correlation**: The standard baseline. It measures linear relationships but is highly sensitive to outliers (e.g., a single volatile minute can artificially pump the correlation near 1.0).
2. **Rolling Spearman Rank Correlation**: A non-parametric measure. By evaluating the *rank* of returns rather than their absolute magnitude, it ignores micro-volatility spikes. Since `pandas.rolling().corr()` doesn't support Spearman directly, we use a highly optimized, vectorized NumPy algorithm (`sliding_window_view`) to compute it instantly over millions of rows.
3. **Directional Hit Ratio (Concordance)**: The most intuitive metric. It calculates the percentage of minutes within the rolling window where both assets moved in the exact same direction (both up, both down, or both flat). This directly answers the question: *"Are these markets actually stepping together?"*

In [3]:
# ==========================================
# CELL 3: ROLLING CORRELATIONS & HIT RATIO
# ==========================================
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

# Define the rolling window size (e.g., 15 minutes)
ROLLING_WINDOW = 15

# 1. Rolling Pearson Correlation
df['corr_pearson'] = df['btc_ret'].rolling(window=ROLLING_WINDOW).corr(df['nq_ret'])

# 2. Fast Vectorized Rolling Spearman Rank Correlation
# We build a custom NumPy engine to calculate Spearman rank correlation.
# Spearman is mathematically identical to the Pearson correlation of ranks.
def fast_rolling_spearman(x, y, window):
    # Create 2D sliding windows out of 1D arrays
    x_win = sliding_window_view(x, window_shape=window)
    y_win = sliding_window_view(y, window_shape=window)
    
    # Calculate ranks (argsort of argsort gives the rank 0 to window-1)
    x_rank = np.argsort(np.argsort(x_win, axis=1), axis=1)
    y_rank = np.argsort(np.argsort(y_win, axis=1), axis=1)
    
    # Since ranks inside every window are always the integers 0 to W-1,
    # the mean and variance of these ranks are mathematically constant:
    rank_mean = (window - 1) / 2.0
    rank_var = (window**2 - 1) / 12.0
    
    # Covariance of the ranks
    covar = np.mean((x_rank - rank_mean) * (y_rank - rank_mean), axis=1)
    
    # Spearman Correlation = Covariance / Variance
    corr = covar / rank_var
    
    # Pad the beginning with NaNs to match the original dataframe length
    return np.concatenate((np.full(window - 1, np.nan), corr))

df['corr_spearman'] = fast_rolling_spearman(df['btc_ret'].values, df['nq_ret'].values, ROLLING_WINDOW)

# 3. Directional Hit Ratio (Concordance)
# Evaluate the sign of returns: +1 (Up), -1 (Down), 0 (Flat)
btc_direction = np.sign(df['btc_ret'])
nq_direction = np.sign(df['nq_ret'])

# Boolean array: 1.0 if directions match, 0.0 if they diverge
directional_match = (btc_direction == nq_direction).astype(float)

# Calculate the rolling mean of matches to get a percentage (0.0 to 1.0)
df['hit_ratio'] = directional_match.rolling(window=ROLLING_WINDOW).mean()

# Drop initial rows that contain NaNs due to the rolling window warmup period
df.dropna(inplace=True)

print(f"Metrics calculated successfully using a {ROLLING_WINDOW}-bar window.")
display(df[['btc_ret', 'nq_ret', 'corr_pearson', 'corr_spearman', 'hit_ratio']].tail(10))

Metrics calculated successfully using a 15-bar window.


,btc_ret,nq_ret,corr_pearson,corr_spearman,hit_ratio
2026-03-06 21:50:00+00:00,0.000331,0.000101,-0.078517,-0.114286,0.533333
2026-03-06 21:51:00+00:00,0.000069,-0.000061,-0.034914,-0.003571,0.466667
2026-03-06 21:52:00+00:00,0.000204,0.000213,-0.112964,-0.071429,0.466667
2026-03-06 21:53:00+00:00,-0.000265,-0.000324,0.244736,0.178571,0.466667
2026-03-06 21:54:00+00:00,-0.000937,0.000182,-0.002812,0.096429,0.466667
2026-03-06 21:55:00+00:00,-0.000522,-0.000142,0.052952,0.153571,0.466667
2026-03-06 21:56:00+00:00,0.000966,-0.000061,-0.001883,0.028571,0.466667
2026-03-06 21:57:00+00:00,0.000099,-0.000091,-0.003487,0.028571,0.466667
2026-03-06 21:58:00+00:00,0.000260,-0.000030,0.017126,0.064286,0.466667
2026-03-06 21:59:00+00:00,-0.000160,-0.000233,0.021132,0.117857,0.466667


## 4. Global Time Series Visualization (EDA)

Interactive visualization is crucial for understanding the dynamic relationship between assets. We use `plotly` to stack the price action directly above our computed correlation metrics. 

**Note on Memory Management:** Since 1-minute data over 2 years produces hundreds of thousands of rows, rendering the entire dataset interactively will crash the browser. We slice the data into a specific time window (e.g., a specific trading week) to observe the micro-structure and evaluate how Pearson, Spearman, and Hit Ratio react to real price movements.

* **Top Panel**: Normalized Prices (Base 100). Allows direct visual comparison of relative momentum.
* **Middle Panel**: Rolling Pearson vs. Rolling Spearman. Notice how Spearman is generally smoother and less reactive to single-bar volatility spikes.
* **Bottom Panel**: Directional Hit Ratio. Shows the percentage of aligned bars. A value of 0.5 implies random walk (no directional correlation), while values > 0.7 or < 0.3 indicate strong regime alignment or divergence.

In [4]:
# ==========================================
# CELL 4: INTERACTIVE VISUALIZATION
# ==========================================

# 1. Define the slice for visualization to prevent browser crashes
# Let's look at the first full trading week in the dataset
START_DATE = '2024-03-11'
END_DATE = '2024-03-15'

plot_df = df.loc[START_DATE:END_DATE].copy()

# 2. Normalize prices to Base 100 for proper visual scale comparison
plot_df['btc_norm'] = (plot_df['btc_close'] / plot_df['btc_close'].iloc[0]) * 100
plot_df['nq_norm'] = (plot_df['nq_close'] / plot_df['nq_close'].iloc[0]) * 100

# 3. Create a multi-panel figure
fig = make_subplots(
    rows=3, cols=1, 
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.5, 0.25, 0.25],
    subplot_titles=(
        "Normalized Prices (Base 100): BTC vs NQ", 
        f"Rolling {ROLLING_WINDOW}-bar Correlations", 
        f"Directional Hit Ratio ({ROLLING_WINDOW}-bar)"
    )
)

# --- ROW 1: Normalized Prices ---
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['btc_norm'], 
                         mode='lines', name='BTC (Base 100)', line=dict(color='#F7931A', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['nq_norm'], 
                         mode='lines', name='NQ (Base 100)', line=dict(color='#00A6DF', width=1.5)), row=1, col=1)

# --- ROW 2: Pearson vs Spearman ---
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['corr_pearson'], 
                         mode='lines', name='Pearson', line=dict(color='#EF553B', width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['corr_spearman'], 
                         mode='lines', name='Spearman', line=dict(color='#00CC96', width=1.5)), row=2, col=1)

# Add a zero-line reference for correlations
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=2, col=1)

# --- ROW 3: Hit Ratio ---
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['hit_ratio'], 
                         mode='lines', name='Hit Ratio', line=dict(color='#AB63FA', width=1)), row=3, col=1)

# Add reference lines for Hit Ratio (0.5 = random, 0.7 = strong directional agreement)
fig.add_hline(y=0.5, line_dash="dash", line_color="gray", row=3, col=1)
fig.add_hline(y=0.7, line_dash="dot", line_color="rgba(0, 204, 150, 0.5)", row=3, col=1)

# 4. Layout formatting
fig.update_layout(
    title=f"Cross-Asset Alignment Analysis ({START_DATE} to {END_DATE})",
    height=900,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_yaxes(title_text="Normalized Value", row=1, col=1)
fig.update_yaxes(title_text="Correlation (-1 to 1)", range=[-1.1, 1.1], row=2, col=1)
fig.update_yaxes(title_text="Hit Ratio (0 to 1)", range=[-0.05, 1.05], row=3, col=1)

fig.show()

## 5. Statistical Summary of Metrics

Visualizing the time series gives us intuition, but quantitative research requires hard statistics. In this cell, we generate a statistical summary table of our correlation metrics across the entire 2-year dataset.

We specifically look at:
* **Mean & Median (50%)**: To establish the baseline "random walk" state.
* **Tails (5%, 95%)**: To understand the extreme thresholds of the metrics.
* **Regime Frequencies**: A quick calculation to see exactly what percentage of the time the markets are in a state of high directional agreement (Hit Ratio $\ge$ 0.70 and $\ge$ 0.80).

In [5]:
# ==========================================
# CELL 5: STATISTICAL SUMMARY TABLE
# ==========================================

# 1. Isolate the metrics for statistical analysis
metrics_df = df[['corr_pearson', 'corr_spearman', 'hit_ratio']]

# 2. Generate summary statistics with custom distribution percentiles
summary_table = metrics_df.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T

# 3. Display the summary table cleanly
print("="*60)
print("GLOBAL STATISTICAL SUMMARY (ENTIRE DATASET)")
print("="*60)
display(summary_table.round(4))

# 4. Calculate specific "High Agreement" frequencies
# How often are BTC and NQ moving in the exact same direction > 70% or > 80% of the time?
hr_gt_70 = (df['hit_ratio'] >= 0.70).mean() * 100
hr_gt_80 = (df['hit_ratio'] >= 0.80).mean() * 100
hr_lt_30 = (df['hit_ratio'] <= 0.30).mean() * 100

print("\n" + "="*60)
print("EXTREME REGIME FREQUENCIES")
print("="*60)
print(f"Time in High Positive Agreement (Hit Ratio >= 70%) : {hr_gt_70:>5.2f}%")
print(f"Time in Extreme Positive Agreement (Hit Ratio >= 80%): {hr_gt_80:>5.2f}%")
print(f"Time in High Negative Divergence (Hit Ratio <= 30%)  : {hr_lt_30:>5.2f}%")

# Base Random Walk Check
median_hr = summary_table.loc['hit_ratio', '50%']
print(f"\nBaseline Market State (Median Hit Ratio): {median_hr:.2f}")
print("*(A value near 0.50 confirms that most of the time, the micro-movements are just uncorrelated noise)*")

GLOBAL STATISTICAL SUMMARY (ENTIRE DATASET)


,count,mean,std,min,5%,25%,50%,75%,95%,max
corr_pearson,681408.0,0.2640,0.3021,-0.9992,-0.2608,0.0596,0.2812,0.4864,0.7318,0.9996
corr_spearman,681408.0,0.2460,0.2929,-0.9107,-0.2571,0.0464,0.2571,0.4607,0.7071,0.9929
hit_ratio,681408.0,0.5665,0.1461,0.0000,0.3333,0.4667,0.6000,0.6667,0.8000,1.0000



EXTREME REGIME FREQUENCIES
Time in High Positive Agreement (Hit Ratio >= 70%) : 18.36%
Time in Extreme Positive Agreement (Hit Ratio >= 80%):  8.44%
Time in High Negative Divergence (Hit Ratio <= 30%)  :  3.40%

Baseline Market State (Median Hit Ratio): 0.60
*(A value near 0.50 confirms that most of the time, the micro-movements are just uncorrelated noise)*


## 6. Intraday Seasonality Analysis (Time of Day)

A common pitfall in cross-asset correlation strategies is falling for "fake" signals generated during low-liquidity periods (e.g., Asian session or early US pre-market). During these times, a single erratic print can artificially pump the standard deviation and rolling correlation.

To filter out this noise, we analyze the average metrics based on the **Time of Day**. 
We temporarily convert the index to New York time (`America/New_York`) to intuitively observe the behavioral shift when Regular Trading Hours (RTH) begin at 09:30 AM EST.

In [6]:
# ==========================================
# CELL 6: INTRADAY SEASONALITY (TIME OF DAY)
# ==========================================

# 1. Create a copy and convert timezone to New York for intuitive trading hours
df_ny = df.copy()
df_ny.index = df_ny.index.tz_convert('America/New_York')

# 2. Extract the hour of the day (0 to 23)
df_ny['hour'] = df_ny.index.hour

# 3. Calculate average metrics per hour
hourly_stats = df_ny.groupby('hour')[['hit_ratio', 'corr_spearman', 'corr_pearson']].mean()

# Calculate what percentage of the "Extreme Regimes" (Hit Ratio >= 0.80) happens in each hour
df_ny['extreme_regime'] = (df_ny['hit_ratio'] >= 0.80).astype(int)
hourly_extreme_freq = df_ny.groupby('hour')['extreme_regime'].sum()
hourly_extreme_pct = (hourly_extreme_freq / hourly_extreme_freq.sum()) * 100

# 4. Visualization
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        "Average Correlation & Hit Ratio by Hour of Day (NY Time)",
        "Distribution of Extreme Agreement Signals (Hit Ratio >= 0.80) by Hour %"
    )
)

# --- ROW 1: Average Metrics ---
fig.add_trace(go.Bar(x=hourly_stats.index, y=hourly_stats['hit_ratio'], 
                     name='Avg Hit Ratio', marker_color='#AB63FA'), row=1, col=1)
fig.add_trace(go.Scatter(x=hourly_stats.index, y=hourly_stats['corr_spearman'], 
                         mode='lines+markers', name='Avg Spearman', line=dict(color='#00CC96', width=3)), row=1, col=1)

# Add reference line for open of US Regular Trading Hours (09:30 AM -> hour 9)
fig.add_vline(x=9.5, line_width=2, line_dash="dash", line_color="white", 
              annotation_text="US Market Open (09:30)", annotation_position="top right", row=1, col=1)

# --- ROW 2: Extreme Signal Frequency ---
fig.add_trace(go.Bar(x=hourly_extreme_pct.index, y=hourly_extreme_pct.values, 
                     name='% of Total Extreme Signals', marker_color='#EF553B'), row=2, col=1)
fig.add_vline(x=9.5, line_width=2, line_dash="dash", line_color="white", row=2, col=1)

# Formatting
fig.update_layout(
    height=800,
    hovermode="x unified",
    xaxis2=dict(title="Hour of the Day (New York Time)", tickmode='linear', tick0=0, dtick=1),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_yaxes(title_text="Metric Value", row=1, col=1)
fig.update_yaxes(title_text="% of Signals", row=2, col=1)

fig.show()

## 6.5 Intraday Seasonality Data Export

To quantitatively validate the visual heatmap from the previous step, we compile the hourly average metrics and the distribution of extreme signals into a single DataFrame. This table is then displayed and exported to a CSV file for further analysis and hardcoding trading session filters.

In [7]:
# ==========================================
# CELL 6.5: EXPORT INTRADAY SEASONALITY DATA
# ==========================================

# Combine the data from Cell 6 into a single summary table
seasonality_table = hourly_stats.copy()
seasonality_table['extreme_signals_pct'] = hourly_extreme_pct

# Round for readability
seasonality_table = seasonality_table.round(4)

# Rename columns for clarity
seasonality_table.columns = [
    'Avg Hit Ratio', 
    'Avg Spearman', 
    'Avg Pearson', 
    '% of Total Extreme Signals'
]

print("="*70)
print("INTRADAY SEASONALITY SUMMARY (NEW YORK TIME)")
print("="*70)
display(seasonality_table)

INTRADAY SEASONALITY SUMMARY (NEW YORK TIME)


,Avg Hit Ratio,Avg Spearman,Avg Pearson,% of Total Extreme Signals
hour,,,,
0,0.5124,0.1531,0.1619,1.6519
1,0.5250,0.1748,0.1885,2.0588
2,0.5362,0.1902,0.2067,2.4501
3,0.5573,0.2240,0.2410,3.2447
4,0.5639,0.2313,0.2463,3.8116
5,0.5487,0.2083,0.2211,3.0134
6,0.5391,0.1955,0.2115,2.6309
7,0.5493,0.2090,0.2263,3.2725
8,0.5655,0.2395,0.2630,3.9455


## 7. Signal Quality Matrix (Spearman vs. Hit Ratio)

To avoid falling into the trap of "mathematically scrunched" correlations during low-liquidity hours, we must evaluate the relationship between continuous correlation (`Spearman`) and discrete bar-by-bar alignment (`Hit Ratio`).

Here we plot a **2D Density Heatmap**. 
* **Toxic Zones**: Areas with high correlation (e.g., Spearman > 0.7) but low Hit Ratio (~0.5). These represent illiquid, erratic movements where math breaks down.
* **Alpha Zones**: Areas where both Spearman is high AND Hit Ratio is high (> 0.7). This is true, structural institutional flow.

In [8]:
# ==========================================
# CELL 7: SIGNAL QUALITY 2D MATRIX
# ==========================================
import plotly.express as px

# We use the New York time dataframe (df_ny)
# To keep the plot extremely fast and avoid overplotting, we randomly sample 10% of the data
# (Statistically, the distribution will remain exactly the same)
sample_df = df_ny.sample(frac=0.10, random_state=42)

# Create a 2D Density Heatmap
fig = px.density_heatmap(
    sample_df, 
    x="corr_spearman", 
    y="hit_ratio", 
    nbinsx=50, 
    nbinsy=20,
    color_continuous_scale="Viridis",
    title="Signal Quality Matrix: Spearman Correlation vs. Directional Hit Ratio"
)

# Add highlight boxes/lines to denote the "Alpha Zone" vs "Toxic Zone"
# Alpha Zone: Spearman > 0.6 AND Hit Ratio >= 0.7
fig.add_shape(type="rect",
    x0=0.6, y0=0.7, x1=1.0, y1=1.0,
    line=dict(color="LimeGreen", width=2, dash="dash"),
    fillcolor="rgba(50, 205, 50, 0.1)"
)
fig.add_annotation(x=0.8, y=0.85, text="ALPHA ZONE<br>(True Flow)", showarrow=False, font=dict(color="LimeGreen"))

# Toxic Zone: Spearman > 0.6 BUT Hit Ratio <= 0.6 (Random walk territory)
fig.add_shape(type="rect",
    x0=0.6, y0=0.3, x1=1.0, y1=0.6,
    line=dict(color="Red", width=2, dash="dash"),
    fillcolor="rgba(255, 0, 0, 0.1)"
)
fig.add_annotation(x=0.8, y=0.45, text="TOXIC ZONE<br>(Fake Pump / Noise)", showarrow=False, font=dict(color="Red"))


fig.update_layout(
    height=700,
    xaxis_title="Rolling Spearman Rank Correlation",
    yaxis_title="Directional Hit Ratio (Concordance)",
    coloraxis_colorbar=dict(title="Density (Observations)")
)

fig.show()

## 7.5 Signal Quality Matrix Data Export

Visual heatmaps are great for intuition, but we need hard percentages to define algorithmic thresholds. This cell creates a cross-tabulation (2D frequency matrix) of our `Hit Ratio` and `Spearman Correlation`. 

It divides the metrics into distinct bins and calculates the percentage of total market time spent in each combination. Most importantly, it calculates the exact weight of our **Alpha Zone** (true directional flow) versus the **Toxic Zone** (mathematical correlation illusion). The matrix is exported to CSV.

In [9]:
# ==========================================
# CELL 7.5: SIGNAL QUALITY MATRIX EXPORT
# ==========================================
import numpy as np

# 1. Define bins for Hit Ratio and Spearman Correlation
hit_ratio_bins = np.arange(0.0, 1.1, 0.1)  # [0.0, 0.1, 0.2 ... 1.0]
spearman_bins = np.arange(-1.0, 1.2, 0.2)  # [-1.0, -0.8, -0.6 ... 1.0]

# 2. Categorize the data into these bins
# We use df_ny (our full dataset with New York timezone)
df_ny['hr_bin'] = pd.cut(df_ny['hit_ratio'], bins=hit_ratio_bins, include_lowest=True)
df_ny['spearman_bin'] = pd.cut(df_ny['corr_spearman'], bins=spearman_bins, include_lowest=True)

# 3. Create a Cross-Tabulation (Frequency Matrix)
# normalize='all' gives us the percentage of total observations, multiplied by 100 for readability
signal_matrix = pd.crosstab(df_ny['hr_bin'], df_ny['spearman_bin'], normalize='all') * 100
signal_matrix = signal_matrix.round(2) # Round to 2 decimal places

# 4. Display the Matrix
print("="*85)
print("SIGNAL QUALITY MATRIX: % OF TOTAL OBSERVATIONS")
print("Rows: Hit Ratio | Columns: Spearman Correlation")
print("="*85)
display(signal_matrix)

# 5. Calculate Specific Zone Weights
# Alpha Zone: True structural flow
alpha_zone_mask = (df_ny['corr_spearman'] >= 0.6) & (df_ny['hit_ratio'] >= 0.7)
alpha_zone_pct = (alpha_zone_mask.sum() / len(df_ny)) * 100

# Toxic Zone: High math correlation, but poor bar-by-bar directional match
toxic_zone_mask = (df_ny['corr_spearman'] >= 0.6) & (df_ny['hit_ratio'] <= 0.6)
toxic_zone_pct = (toxic_zone_mask.sum() / len(df_ny)) * 100

print("\n" + "="*85)
print("ZONE ANALYSIS (KEY THRESHOLDS)")
print("="*85)
print(f"[ALPHA ZONE] High Spearman (>=0.6) & High Hit Ratio (>=0.7): {alpha_zone_pct:.2f}% of all market time")
print(f"[TOXIC ZONE] High Spearman (>=0.6) & Low Hit Ratio (<=0.6) : {toxic_zone_pct:.2f}% of all market time")

SIGNAL QUALITY MATRIX: % OF TOTAL OBSERVATIONS
Rows: Hit Ratio | Columns: Spearman Correlation


spearman_bin,"(-1.001, -0.8]","(-0.8, -0.6]","(-0.6, -0.4]","(-0.4, -0.2]","(-0.2, -2.22e-16]","(-2.22e-16, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
hr_bin,,,,,,,,,,
"(-0.001, 0.1]",0.0,0.01,0.02,0.01,0.01,0.00,0.00,0.00,0.00,0.00
"(0.1, 0.2]",0.0,0.08,0.29,0.38,0.20,0.06,0.02,0.00,0.00,0.00
"(0.2, 0.3]",0.0,0.03,0.34,0.84,0.76,0.28,0.05,0.00,0.00,0.00
"(0.3, 0.4]",0.0,0.03,0.62,2.74,5.17,4.41,1.60,0.21,0.01,0.00
"(0.4, 0.5]",0.0,0.01,0.11,0.98,3.36,5.32,3.56,0.91,0.05,0.00
"(0.5, 0.6]",0.0,0.00,0.05,0.63,3.42,9.38,12.57,7.20,1.36,0.03
"(0.6, 0.7]",0.0,0.00,0.00,0.03,0.35,1.75,4.71,5.43,2.15,0.10
"(0.7, 0.8]",0.0,0.00,0.00,0.00,0.07,0.63,2.75,5.99,5.09,0.77
"(0.8, 0.9]",0.0,0.00,0.00,0.00,0.00,0.01,0.09,0.52,1.14,0.49



ZONE ANALYSIS (KEY THRESHOLDS)
[ALPHA ZONE] High Spearman (>=0.6) & High Hit Ratio (>=0.7): 8.21% of all market time
[TOXIC ZONE] High Spearman (>=0.6) & Low Hit Ratio (<=0.6) : 1.45% of all market time


## 8. Signal Generation & Event Clustering

Using the insights from our EDA, we now define the hard rules for entering a trade. 
A valid **Regime Signal** occurs when:
1. **Time Filter**: Between 09:30 and 16:00 NY Time (Regular Trading Hours).
2. **Directional Flow**: Hit Ratio >= 0.70.
3. **Mathematical Validation**: Spearman Correlation >= 0.60.

Because these conditions will persist over multiple consecutive 1-minute bars, we group adjacent signal bars into **Signal Clusters (Events)**. This tells us exactly how many distinct trading opportunities occurred over the 2-year period.

In [10]:
# ==========================================
# CELL 8: SIGNAL LOGIC & CLUSTERING
# ==========================================

# Use the New York timezone dataframe
signal_df = df_ny.copy()

# 1. Apply Hard Filters
time_filter = (signal_df.index.time >= pd.to_datetime('09:30').time()) & \
              (signal_df.index.time <= pd.to_datetime('16:00').time())

flow_filter = signal_df['hit_ratio'] >= 0.70
math_filter = signal_df['corr_spearman'] >= 0.60

# Combined Boolean Series: True if ALL conditions are met
signal_df['valid_signal'] = time_filter & flow_filter & math_filter

# 2. Cluster consecutive signals into "Events"
# We want to count distinct periods of alignment, not just individual minutes
# A new event starts if the previous bar was NOT a signal, but the current one IS
signal_df['event_start'] = (signal_df['valid_signal']) & (~signal_df['valid_signal'].shift(1).fillna(False))

# Calculate total number of distinct events
total_events = signal_df['event_start'].sum()

print("="*60)
print("REGIME SIGNAL ANALYSIS")
print("="*60)
print(f"Total evaluated 1-min bars: {len(signal_df):,}")
print(f"Total minutes meeting ALL conditions: {signal_df['valid_signal'].sum():,}")
print(f"Number of distinct Trading Events (Clusters): {total_events:,}")

# Let's see on average how many events happen per trading day
trading_days = len(np.unique(signal_df.index.date))
events_per_day = total_events / trading_days

print(f"Total Trading Days in Dataset: {trading_days:,}")
print(f"Average Actionable Events per Day: {events_per_day:.2f}")

# 3. Extract the exact timestamps of the first 5 events to see what they look like
print("\nFirst 5 Signal Event Timestamps:")
display(signal_df[signal_df['event_start']][['hit_ratio', 'corr_spearman']].head(5))

REGIME SIGNAL ANALYSIS
Total evaluated 1-min bars: 681,408
Total minutes meeting ALL conditions: 35,020
Number of distinct Trading Events (Clusters): 6,116
Total Trading Days in Dataset: 624
Average Actionable Events per Day: 9.80

First 5 Signal Event Timestamps:


,hit_ratio,corr_spearman
2024-03-08 12:05:00-05:00,0.866667,0.667857
2024-03-11 11:26:00-04:00,0.933333,0.646429
2024-03-11 11:35:00-04:00,0.800000,0.600000
2024-03-11 11:37:00-04:00,0.800000,0.617857
2024-03-11 13:12:00-04:00,0.866667,0.632143


## 9. Forward Returns Analysis (Event Study)

Identifying a statistical regime is only half the battle. The ultimate question for a Quant is: *"Does the price actually follow through after the signal is triggered?"*

In this final cell, we conduct a **Forward Returns Analysis**. We:
1. Determine the prevailing direction of the regime (Up or Down) using the 15-minute momentum at the time of the event.
2. Look ahead exactly 5, 10, and 15 minutes after the `event_start` trigger.
3. Calculate the directional returns. A **positive percentage** indicates trend continuation (profit), while a negative percentage indicates a reversal (loss).

In [11]:
# ==========================================
# CELL 9: FORWARD RETURNS (EVENT STUDY)
# ==========================================

# 1. Calculate future returns in the main dataframe
horizons = [5, 10, 15]

for h in horizons:
    # Future Return = (Future Price / Current Price) - 1
    # Note: .shift(-h) looks forward in time
    signal_df[f'nq_fwd_{h}m'] = signal_df['nq_close'].shift(-h) / signal_df['nq_close'] - 1
    signal_df[f'btc_fwd_{h}m'] = signal_df['btc_close'].shift(-h) / signal_df['btc_close'] - 1

# 2. Determine the prevailing trend direction at the moment of the signal
# We use the 15-minute momentum of NQ as the regime compass
signal_df['mom_15m'] = signal_df['nq_close'] / signal_df['nq_close'].shift(ROLLING_WINDOW) - 1
signal_df['regime_dir'] = np.sign(signal_df['mom_15m'])
signal_df['regime_dir'] = signal_df['regime_dir'].replace(0, 1) # Fallback for flat momentum

# 3. Extract ONLY the Event Start moments
events_df = signal_df[signal_df['event_start']].copy()

# 4. Calculate Directional Forward Returns (in Basis Points for readability, 1 bp = 0.01%)
# Directional Return = Raw Return * Regime Direction
# Positive result = Trend continued (Win). Negative result = Trend reversed (Loss).
for h in horizons:
    events_df[f'nq_dir_fwd_{h}m_bps'] = events_df[f'nq_fwd_{h}m'] * events_df['regime_dir'] * 10000
    events_df[f'btc_dir_fwd_{h}m_bps'] = events_df[f'btc_fwd_{h}m'] * events_df['regime_dir'] * 10000

# 5. Summarize the Results
fwd_cols = [c for c in events_df.columns if 'dir_fwd' in c]
forward_summary = events_df[fwd_cols].describe(percentiles=[0.25, 0.5, 0.75]).T

print("="*80)
print("AVERAGE DIRECTIONAL FORWARD RETURNS (IN BASIS POINTS)")
print("Note: Positive mean/median means the trend successfully continued after the signal.")
print("10 Basis Points (bps) = 0.1%")
print("="*80)
display(forward_summary[['count', 'mean', 'std', '50%']].round(2))

# 6. Win Rate Calculation
# What percentage of trades would be profitable if held for exactly X minutes?
print("\n" + "="*80)
print("TIME-BASED WIN RATES (% of signals with positive follow-through)")
print("="*80)
for h in horizons:
    nq_win_rate = (events_df[f'nq_dir_fwd_{h}m_bps'] > 0).mean() * 100
    btc_win_rate = (events_df[f'btc_dir_fwd_{h}m_bps'] > 0).mean() * 100
    print(f"Holding for {h:>2} mins | NQ Win Rate: {nq_win_rate:.1f}% | BTC Win Rate: {btc_win_rate:.1f}%")

AVERAGE DIRECTIONAL FORWARD RETURNS (IN BASIS POINTS)
Note: Positive mean/median means the trend successfully continued after the signal.
10 Basis Points (bps) = 0.1%


,count,mean,std,50%
nq_dir_fwd_5m_bps,6116.0,0.23,13.17,0.00
btc_dir_fwd_5m_bps,6116.0,0.28,23.26,-0.12
nq_dir_fwd_10m_bps,6116.0,0.54,19.91,0.24
btc_dir_fwd_10m_bps,6116.0,0.56,32.70,0.04
nq_dir_fwd_15m_bps,6116.0,0.47,23.20,0.12
btc_dir_fwd_15m_bps,6116.0,0.50,40.79,-0.54



TIME-BASED WIN RATES (% of signals with positive follow-through)
Holding for  5 mins | NQ Win Rate: 50.0% | BTC Win Rate: 49.6%
Holding for 10 mins | NQ Win Rate: 50.6% | BTC Win Rate: 50.1%
Holding for 15 mins | NQ Win Rate: 50.2% | BTC Win Rate: 49.2%


## 10. Multi-Dimensional Sensitivity Analysis (Grid Search)

To ensure our Regime Gate is robust, we cannot rely on a single parameter. Here, we conduct a multi-dimensional grid search. 
We evaluate the interaction between **Regime Window Size** (how long it takes to establish structural flow) and **Holding Horizon** (how long the momentum persists before mean-reversion).

We will output two styled matrices (Heatmaps):
1. **Win Rate Matrix**: Shows the percentage of trades that had positive follow-through.
2. **Forward Returns Matrix**: Shows the average profitability in Basis Points (bps).

*Note: Since the strategy intends to use a Volatility Breakout (Fat Tail / SD > 2) as the final entry trigger, this matrix helps us understand the baseline environment into which that breakout will fire.*

In [13]:
# ==========================================
# CELL 10: ROBUST SENSITIVITY GRID SEARCH
# ==========================================
import pandas as pd
import numpy as np

# 1. Define the grid space
test_windows = [5, 10, 15, 20, 30, 45, 60]  # Lookback for correlation & hit ratio
hold_horizons = [5, 10, 15, 20, 30]         # Minutes to look ahead

results_wr = {}
results_bps = {}
event_counts = {}

print("Running Multi-Dimensional Grid Search... Please wait.")

# 2. Iterate through different Regime Window Sizes
for w in test_windows:
    temp_df = df_ny[['btc_ret', 'nq_ret', 'nq_close']].copy()
    
    # Recalculate metrics for the current window 'w'
    temp_df['corr_spearman'] = fast_rolling_spearman(temp_df['btc_ret'].values, temp_df['nq_ret'].values, w)
    match = (np.sign(temp_df['btc_ret']) == np.sign(temp_df['nq_ret'])).astype(float)
    temp_df['hit_ratio'] = match.rolling(window=w).mean()
    
    # Identify signal states (RTH time + correlation thresholds)
    time_filter = (temp_df.index.time >= pd.to_datetime('09:30').time()) & \
                  (temp_df.index.time <= pd.to_datetime('16:00').time())
    
    temp_df['valid_signal'] = time_filter & (temp_df['hit_ratio'] >= 0.70) & (temp_df['corr_spearman'] >= 0.60)
    temp_df['event_start'] = (temp_df['valid_signal']) & (~temp_df['valid_signal'].shift(1).fillna(False))
    
    num_events = temp_df['event_start'].sum()
    event_counts[w] = num_events
    
    # Calculate regime direction (momentum over the window 'w')
    temp_df['mom_win'] = temp_df['nq_close'] / temp_df['nq_close'].shift(w) - 1
    temp_df['regime_dir'] = np.sign(temp_df['mom_win']).replace(0, 1)
    
    # Calculate all forward horizons on the whole dataframe FIRST (before filtering)
    for h in hold_horizons:
        temp_df[f'nq_fwd_{h}'] = temp_df['nq_close'].shift(-h) / temp_df['nq_close'] - 1
        
    # Isolate only the rows where an event started
    events = temp_df[temp_df['event_start']].copy()
    
    wr_row = {}
    bps_row = {}
    
    # Evaluate performance for each holding horizon
    for h in hold_horizons:
        events[f'nq_dir_fwd_{h}'] = events[f'nq_fwd_{h}'] * events['regime_dir']
        
        if num_events > 0:
            win_rate = (events[f'nq_dir_fwd_{h}'] > 0).mean() * 100
            avg_bps = events[f'nq_dir_fwd_{h}'].mean() * 10000
        else:
            win_rate, avg_bps = np.nan, np.nan
            
        wr_row[f'Hold_{h}m'] = win_rate
        bps_row[f'Hold_{h}m'] = avg_bps
        
    results_wr[w] = wr_row
    results_bps[w] = bps_row

# 3. Build and format output DataFrames
df_wr = pd.DataFrame.from_dict(results_wr, orient='index')
df_wr.index.name = 'Window_Bars'
df_wr.insert(0, 'Total_Events', pd.Series(event_counts))

df_bps = pd.DataFrame.from_dict(results_bps, orient='index')
df_bps.index.name = 'Window_Bars'
df_bps.insert(0, 'Total_Events', pd.Series(event_counts))

print("\n" + "="*80)
print("WIN RATE MATRIX (%) - Dark Green = Higher Win Rate")
print("="*80)
display(df_wr.style.background_gradient(cmap='RdYlGn', subset=df_wr.columns[1:], axis=None).format(precision=2))

print("\n" + "="*80)
print("AVERAGE FORWARD RETURNS MATRIX (Basis Points) - Dark Green = Higher PnL")
print("="*80)
display(df_bps.style.background_gradient(cmap='RdYlGn', subset=df_bps.columns[1:], axis=None).format(precision=2))

Running Multi-Dimensional Grid Search... Please wait.

WIN RATE MATRIX (%) - Dark Green = Higher Win Rate


,Total_Events,Hold_5m,Hold_10m,Hold_15m,Hold_20m,Hold_30m
Window_Bars,,,,,,
5,15888,49.90,49.52,49.92,50.36,50.39
10,9796,50.49,51.34,50.90,50.73,49.97
15,6116,49.98,50.59,50.18,49.85,49.62
20,4968,50.00,50.12,50.70,51.09,50.16
30,3173,49.80,50.80,49.48,48.28,49.16
45,1946,52.16,52.67,53.08,51.49,52.11
60,1421,48.63,49.47,50.39,51.23,50.67



AVERAGE FORWARD RETURNS MATRIX (Basis Points) - Dark Green = Higher PnL


,Total_Events,Hold_5m,Hold_10m,Hold_15m,Hold_20m,Hold_30m
Window_Bars,,,,,,
5,15888,0.15,0.27,0.22,0.43,0.40
10,9796,0.25,0.56,0.45,0.26,0.05
15,6116,0.23,0.54,0.47,0.44,0.35
20,4968,0.10,0.23,0.46,0.42,0.29
30,3173,0.26,0.33,0.12,-0.11,-0.70
45,1946,0.62,0.72,0.61,0.54,0.21
60,1421,-0.14,0.10,0.17,0.58,0.78
